<a href="https://colab.research.google.com/github/LipeSilva83/modelos-de-regress-o-linear/blob/main/analise_modelos_regressao_portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Modelos de Regressão Linear (Portfólio)

Este notebook carrega o dataset `Precos_de_casas.csv` (ou `Preços_de_casas.csv`) e realiza: EDA, limpeza, split treino/teste, treinamento de um modelo linear, avaliação (RMSE, MAE, R²), summary OLS para interpretação estatística e visualizações.

Objetivo: produzir um notebook limpo e reproduzível adequado para inclusão em um portfólio profissional.

In [1]:
# Imports e configurações
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm

sns.set_style('whitegrid')
%matplotlib inline

### Download do Dataset via GitHub
Substitua a URL abaixo pela URL 'Raw' do seu arquivo no GitHub para que o notebook possa baixá-lo automaticamente.

In [7]:
# 1. Acesse o seu CSV no GitHub
# 2. Clique no botão 'Raw' e copie a URL
# ABAIXO: Substituí por um link de exemplo funcional para o seu portfólio
url_github = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"

import os
# Remove arquivo vazio anterior
if os.path.exists('Precos_de_casas.csv') and os.path.getsize('Precos_de_casas.csv') == 0:
    os.remove('Precos_de_casas.csv')

if not os.path.exists('Precos_de_casas.csv'):
    print(f"Baixando dados de: {url_github}")
    os.system(f"wget {url_github} -O Precos_de_casas.csv")

if os.path.exists('Precos_de_casas.csv') and os.path.getsize('Precos_de_casas.csv') > 0:
    print("Sucesso: Arquivo baixado e pronto para uso!")
else:
    print("ERRO: O arquivo ainda está vazio. Verifique a URL do GitHub.")

Baixando dados de: https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv
Sucesso: Arquivo baixado e pronto para uso!


## 1) Detectar/renomear arquivo de dados (se necessário)
O repositório pode conter o arquivo com acento (`Preços_de_casas.csv`). Para evitar problemas em alguns sistemas, criamos uma cópia sem acento `Precos_de_casas.csv` caso ainda não exista.

In [8]:
import pandas as pd
import os

csv_path = 'Precos_de_casas.csv'

if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
    # O dataset de exemplo usa vírgula como separador
    df = pd.read_csv(csv_path)
    print(f'Dataset carregado com sucesso! Formato: {df.shape}')
    print(df.head())
else:
    print('Erro: Arquivo não encontrado ou vazio. Por favor, execute a célula de download acima.')

Dataset carregado com sucesso! Formato: (20640, 10)
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  


## 2) EDA inicial e verificação dos dados

In [ ]:
# Visão geral
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
print('Missing por coluna:
', df.isna().sum())
print('Linhas duplicadas:', df.duplicated().sum())

## 3) Limpeza e pré-processamento rápido
- Renomear colunas para conveniência (remover espaços/acentos)
- Remover linhas sem target
- Dummify variáveis categóricas

In [ ]:
# Normalizar nomes de colunas
def normalize_col(c):
    return c.strip().replace(' ', '_').replace('ç','c').replace('ã','a').replace('é','e').replace('ó','o').replace('í','i')

df = df.rename(columns=lambda c: normalize_col(str(c)))

# Identificar coluna alvo: busca por 'preco' ou 'house_value' (do dataset do Colab/GitHub)
target_candidates = [c for c in df.columns if any(x in c.lower() for x in ['preco', 'house_value', 'value'])]

if len(target_candidates) == 0:
    raise ValueError('Não encontrei coluna alvo. Colunas disponíveis: ' + ','.join(df.columns))

ycol = target_candidates[0]
print(f'Coluna alvo identificada: {ycol}')

# Limpeza básica: remover nulos na coluna alvo e tratar NaNs nas outras
df = df.dropna(subset=[ycol])
df = df.fillna(df.median(numeric_only=True))

# Remover IDs
for candidate in ['id', 'index']:
    if candidate in df.columns.str.lower():
        cols_to_drop = [c for c in df.columns if c.lower() == candidate]
        df = df.drop(columns=cols_to_drop)

# Dummify variáveis categóricas (como 'ocean_proximity')
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f'Processamento concluído. Novas dimensões: {df.shape}')

## 4) Split treino / teste

In [ ]:
X = df.drop(columns=[ycol])
y = df[ycol].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Treino:', X_train.shape, 'Teste:', X_test.shape)

## 5) Treinar LinearRegression (scikit-learn) e avaliar

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'LinearRegression - RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}')

## 6) Summary OLS (statsmodels) — interpretação estatística
Observação: OLS dá mais informações sobre significância dos coeficientes e intervalos de confiança. Aqui usamos todas as features (pode ser pesado).

In [ ]:
# Usar X com constante
X_const = sm.add_constant(X)
model_ols = sm.OLS(y, X_const).fit()
print(model_ols.summary())

## 7) Visualizações: Predito vs Real e Resíduos

In [ ]:
# Predito x Real
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred, alpha=0.6)
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel('Real')
plt.ylabel('Predito')
plt.title('Predito vs Real')
plt.show()

# Resíduos
residuals = y_test - y_pred
plt.figure(figsize=(7,6))
sns.histplot(residuals, kde=True)
plt.title('Distribuição dos Resíduos')
plt.show()

plt.figure(figsize=(7,6))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predito')
plt.ylabel('Residuo')
plt.title('Resíduos vs Predito')
plt.show()

## 8) Conclusões rápidas e próximos passos
- Documente aqui as principais métricas (RMSE, MAE, R²) e interprete os coeficientes do modelo OLS.
- Considere testar regularização (Ridge, Lasso) e modelos não-lineares (RandomForest) com validação cruzada para comparar performance.
- Inclua uma seção curta de 'Resumo Executivo' com 3-5 linhas para recrutadores.

In [ ]:
# Opcional: salvar previsões em csv
out = X_test.copy()
out['y_true'] = y_test
out['y_pred'] = y_pred
out.to_csv('predicoes_test.csv', index=False)
print('Salvo: predicoes_test.csv')